In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


In [2]:
knowledge_base = [
    "You can reset your password using the forgot password link.",
    "Billing invoices are available in your account dashboard.",
    "Users can update their email address from account settings.",
    "Login issues can occur due to incorrect credentials.",
    "Two-factor authentication improves account security.",
    "Subscription plans can be upgraded at any time.",
    "Refund requests are processed within five business days.",
    "Account deletion permanently removes all user data.",
    "You can contact support through the help center.",
    "Profile information can be edited from the profile page."
]

In [3]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/home/nineleaps/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
embeddings = model.encode(
    knowledge_base
)

print("\nEmbedding Matrix Shape:")
print(embeddings.shape)



Embedding Matrix Shape:
(10, 384)


In [5]:
embeddings = embeddings.astype("float32")

faiss.normalize_L2(
    embeddings
)

In [6]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    embeddings
)

print("\nVectors Stored:")
print(index.ntotal)


Vectors Stored:
10


In [7]:
while True:

    query = input(
        "\nEnter Query (or 'exit'): "
    )

    if query.lower() == "exit":
        print("Goodbye!")
        break

    query_embedding = model.encode(
        [query]
    )

    query_embedding = query_embedding.astype(
        "float32"
    )

    faiss.normalize_L2(
        query_embedding
    )

    distances, indices = index.search(
        query_embedding,
        3
    )

    print(
        "\nTop 3 Matches"
    )

    print(
        "-" * 70
    )

    print(
        f"{'Rank':<10}{'Score':<15}{'Matched Sentence'}"
    )

    print(
        "-" * 70
    )

    for rank, (distance, idx) in enumerate(
        zip(
            distances[0],
            indices[0]
        ),
        start=1
    ):

        print(
            f"{rank:<10}"
            f"{distance:<15.4f}"
            f"{knowledge_base[idx]}"
        )


Enter Query (or 'exit'):  How can I change my password



Top 3 Matches
----------------------------------------------------------------------
Rank      Score          Matched Sentence
----------------------------------------------------------------------
1         0.6599         You can reset your password using the forgot password link.
2         1.3011         Login issues can occur due to incorrect credentials.
3         1.3146         Users can update their email address from account settings.



Enter Query (or 'exit'):  where can I use my invoices?



Top 3 Matches
----------------------------------------------------------------------
Rank      Score          Matched Sentence
----------------------------------------------------------------------
1         0.4827         Billing invoices are available in your account dashboard.
2         1.5914         You can contact support through the help center.
3         1.7272         Subscription plans can be upgraded at any time.



Enter Query (or 'exit'):  exit


Goodbye!


# FAISS Theory Answers

## Q1: What is the difference between IndexFlatL2 and IndexFlatIP in FAISS? When would you use each?

### IndexFlatL2
- Uses Euclidean Distance (L2 Distance).
- Smaller distance means vectors are more similar.
- Suitable when Euclidean distance is the desired similarity metric.

### IndexFlatIP
- Uses Inner Product (Dot Product).
- Larger values indicate greater similarity.
- Commonly used when embeddings are normalized and cosine similarity is desired.

### When to Use

Use IndexFlatL2:
- When Euclidean distance is required.

Use IndexFlatIP:
- When working with normalized embeddings and cosine similarity.

---

## Q2: Why do we normalize embeddings before adding to FAISS when we want cosine similarity?

Cosine similarity measures the angle between two vectors rather than their magnitude.

Normalization converts vectors to unit length.

After normalization:

Cosine Similarity:

cos(A,B)

becomes proportional to the dot product:

A · B

This allows FAISS to efficiently perform cosine similarity search.

Without normalization, vector magnitude would influence similarity scores.

---

## Q3: FAISS uses ANN (Approximate Nearest Neighbour) search. What does "approximate" mean here and why is it acceptable?

Approximate means FAISS may not always return the exact nearest neighbour.

Instead, it returns results that are extremely close while being much faster and more memory efficient.

Advantages:

- Faster search
- Lower memory usage
- Scales to millions or billions of vectors

Why acceptable?

In semantic search and RAG systems, finding a very close match is usually sufficient.

The small loss in accuracy is outweighed by the significant gain in speed and scalability.